# ERGT Phase 3 Ratio-Matched T0.20

Run this notebook on Google Colab with `Runtime > Change runtime type > GPU`.

This is the decisive light control after Stable Base: it compares `real_d`, `random_d`, and `shuffled_d` at matched `geo_to_qk_ratio = 0.20`, so the geometric term has approximately equal strength across conditions.

In [ ]:
REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
BRANCH = "main"
PROJECT_DIR = "/content/ERGT"
DATA_DIR = "data/processed/wikitext2_gpt2_ctx256"
DEVICE = "cuda"
TARGET_RATIO = 0.20
RATIO_TOLERANCE = 0.03
FORCE_RERUN = False

BASELINE_RESULT = "runs/phase0_baseline/phase3_stable_base_seed2027/baseline_results.json"
RATIO_OUTPUT_DIR = "runs/phase3_geo_attention/phase3_ratio_matched"
COMPARISON_DIR = f"{RATIO_OUTPUT_DIR}/target_0_2_comparison"

RATIO_RUNS = {
    "real_d_target_0_2": {
        "family": "real_d",
        "config": "configs/ergt_v1/phase3_ratio_matched/target_0_2/real_d_target_0_2_alpha_0_0682799_seed2027.json",
        "result": f"{RATIO_OUTPUT_DIR}/target_0_2/real_d_target_0_2_alpha_0_0682799/metrics.json",
    },
    "random_d_target_0_2": {
        "family": "random_d",
        "config": "configs/ergt_v1/phase3_ratio_matched/target_0_2/random_d_target_0_2_alpha_0_103632_seed2027.json",
        "result": f"{RATIO_OUTPUT_DIR}/target_0_2/random_d_target_0_2_alpha_0_103632/metrics.json",
    },
    "shuffled_d_target_0_2": {
        "family": "shuffled_d",
        "config": "configs/ergt_v1/phase3_ratio_matched/target_0_2/shuffled_d_target_0_2_alpha_0_0780132_seed2027.json",
        "result": f"{RATIO_OUTPUT_DIR}/target_0_2/shuffled_d_target_0_2_alpha_0_0780132/metrics.json",
    },
}

## 1. Runtime probe

In [ ]:
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("platform:", platform.platform())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Set Colab hardware accelerator to GPU.")
print("cuda_device_name:", torch.cuda.get_device_name(0))

## 2. Clone/update repo and install dependencies

In [ ]:
def run_command(command, cwd=PROJECT_DIR):
    print("\n$", " ".join(map(str, command)), flush=True)
    subprocess.run(command, cwd=str(cwd), check=True)


project_path = Path(PROJECT_DIR)
if project_path.exists():
    run_command(["git", "fetch", "origin", BRANCH])
    run_command(["git", "checkout", BRANCH])
    run_command(["git", "pull", "--ff-only", "origin", BRANCH])
else:
    run_command(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], cwd=Path("/content"))

os.chdir(PROJECT_DIR)
repo_head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("repo_head:", repo_head)
run_command([sys.executable, "-m", "pip", "install", "-q", "-e", ".[data,viz]"])

## 3. Prepare WikiText-2

In [ ]:
run_command(
    [
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ]
)

## 4. Helpers

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def load_json(path):
    with (Path(PROJECT_DIR) / path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path):
    path = Path(PROJECT_DIR) / path
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def progress_path(result_path):
    return str(Path(result_path).parent / "progress_log.jsonl")


def format_cell(value, digits=4):
    if value is None:
        return "-"
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return str(value)
    if not math.isfinite(numeric):
        return "-"
    return f"{numeric:.{digits}f}" if abs(numeric) < 1000 else f"{numeric:.0f}"


def latest_progress(label, result_path):
    rows = load_jsonl(progress_path(result_path))
    if not rows:
        return {"condition": label, "status": "missing"}
    last = dict(rows[-1])
    last["condition"] = label
    last["status"] = "ok"
    return last


def display_progress_table(run_map):
    rows = [latest_progress(label, path) for label, path in run_map.items()]
    keys = [
        "condition",
        "step",
        "validation_loss",
        "best_validation_loss",
        "perplexity",
        "alpha_effective",
        "geo_to_qk_ratio",
        "tokens_per_second",
        "gpu_memory_gb",
    ]
    headers = ["condition", "step", "val", "best", "ppl", "alpha", "geo/qk", "tok/s", "gpu GB"]
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(format_cell(row.get(key)) for key in keys) + " |")
    display(Markdown("\n".join(lines)))


def plot_ratio_progress(run_map):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4), constrained_layout=True)
    val_axis, ratio_axis, bar_axis = axes
    final_losses = []

    for label, result_path in run_map.items():
        rows = [row for row in load_jsonl(progress_path(result_path)) if row.get("validation_loss") is not None]
        if not rows:
            continue
        steps = [row["step"] for row in rows]
        val_axis.plot(steps, [row["validation_loss"] for row in rows], marker="o", label=label)
        ratio_axis.plot(steps, [row.get("geo_to_qk_ratio") for row in rows], marker="o", label=label)
        final_losses.append((label, rows[-1]["validation_loss"]))

    val_axis.set_title("validation_loss")
    ratio_axis.set_title("geo_to_qk_ratio")
    ratio_axis.axhline(TARGET_RATIO, color="black", linestyle="--", linewidth=1.0)
    ratio_axis.set_ylabel("target is 0.20")

    if final_losses:
        labels = [label for label, _loss in final_losses]
        losses = [loss for _label, loss in final_losses]
        bar_axis.bar(labels, losses)
        bar_axis.set_title("final validation_loss")
        bar_axis.tick_params(axis="x", rotation=25)

    for axis in axes:
        axis.set_xlabel("step")
        axis.grid(True, alpha=0.25)
    val_axis.legend(fontsize=8)
    ratio_axis.legend(fontsize=8)
    plt.show()

## 5. Run ratio-matched target 0.20

In [ ]:
completed = {}
for label, spec in RATIO_RUNS.items():
    result_path = Path(PROJECT_DIR) / spec["result"]
    if result_path.exists() and not FORCE_RERUN:
        print("Skipping existing ratio-matched run:", label, spec["result"])
    else:
        print("\n=== Ratio-matched condition:", label, "===", flush=True)
        run_command(
            [
                sys.executable,
                "experiments/train_ergt_v1.py",
                "--config",
                spec["config"],
                "--data-dir",
                DATA_DIR,
                "--device",
                DEVICE,
            ]
        )
    completed[label] = spec["result"]
    display(Markdown(f"### Progress after `{label}`"))
    display_progress_table(completed)
    plot_ratio_progress(completed)

for label, spec in RATIO_RUNS.items():
    if not (Path(PROJECT_DIR) / spec["result"]).exists():
        raise FileNotFoundError(f"Missing ratio-matched result for {label}: {spec['result']}")

## 6. Compare ratio-matched results

In [ ]:
command = [
    sys.executable,
    "experiments/compare_phase3_ratio_matched.py",
    "--baseline",
    BASELINE_RESULT,
    "--ratio-tolerance",
    str(RATIO_TOLERANCE),
    "--output-dir",
    COMPARISON_DIR,
]
for spec in RATIO_RUNS.values():
    command.extend(["--run", f"{spec['family']}:{TARGET_RATIO}:{spec['result']}"])
run_command(command)

report_path = Path(COMPARISON_DIR) / "phase3_ratio_matched_results.json"
report = load_json(str(report_path))
print("\n=== FINAL RATIO-MATCHED RESULT ===")
print(json.dumps(report["summary"], indent=2, sort_keys=True))
print("\n=== CHECKS ===")
print(json.dumps(report["checks"], indent=2, sort_keys=True))
print("\n=== RANKING ===")
print(json.dumps(report["ranking"], indent=2, sort_keys=True))

## 7. Archive light outputs

In [ ]:
zip_path = Path("/content/ergt_phase3_ratio_matched_t02_light.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for base in [
        Path(PROJECT_DIR) / "configs/ergt_v1/phase3_ratio_matched",
        Path(PROJECT_DIR) / RATIO_OUTPUT_DIR,
    ]:
        for path in base.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(PROJECT_DIR))

print("Light archive ready:", zip_path)
try:
    from google.colab import files

    files.download(str(zip_path))
except Exception as exc:
    print("Manual download path:", zip_path, "download_error:", repr(exc))